# Day 25: SLMs and Token Optimisation
**Small language models vs an API LLM on SmartDesk tickets, and the levers that cut token cost**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Install
For the small model, use a **GPU runtime** in Colab: *Runtime > Change runtime type > GPU*. No GPU? The notebook falls back to a smaller model on CPU. No downloads possible? It uses a clearly labelled offline stand-in so every cell still runs.

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai tiktoken scikit-learn numpy pandas matplotlib transformers accelerate huggingface_hub

# ===== Option B: VS Code / local =====
# pip install openai tiktoken scikit-learn numpy pandas matplotlib transformers accelerate huggingface_hub torch
# Create a .env file (git-ignored) or set the variable in your terminal:  OPENAI_API_KEY=...
# Some small models need a free Hugging Face account: huggingface-cli login (optional)
print("Setup done")

## Setup and API key
With an `OPENAI_API_KEY` the notebook calls a real model. Without one, it runs in **offline demo mode** using a tiny stand-in, so every cell still works and you can study the code. Offline scores show how the code works, not real model quality.

The model name comes from the `LLM_MODEL` environment variable. The default is the program's approved default; confirm it against the approved model list in `docs/MODEL_GUIDE.md`.

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
usage() -> token counts of the last llm() call (estimated offline).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")
LAST_USAGE = {"input_tokens": 0, "output_tokens": 0}

_ENC = []

def count_tokens(text):
    if not _ENC:
        try:
            import tiktoken
            _ENC.append(tiktoken.get_encoding("o200k_base"))
        except Exception:
            _ENC.append(None)                        # no tiktoken or no download: use the estimate
    if _ENC[0] is not None:
        return len(_ENC[0].encode(text))
    return max(1, round(len(text) / 4))              # rough rule: about 4 characters per token

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "classify" in low:                            # SmartDesk ticket categories
        last = low.strip().splitlines()[-1]
        for label, words in [("billing", ["bill", "charge", "refund", "invoice", "payment"]),
                             ("network", ["signal", "network", "slow", "outage", "data", "5g"]),
                             ("account", ["password", "login", "account", "sim", "plan"])]:
            if any(w in last for w in words):
                return label
        return "other"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None, max_tokens=None):
    if ONLINE:
        kwargs = {"max_tokens": max_tokens} if max_tokens else {}
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}], **kwargs)
        LAST_USAGE.update(input_tokens=r.usage.prompt_tokens, output_tokens=r.usage.completion_tokens)
        return r.choices[0].message.content
    out = offline_llm(prompt, system)
    LAST_USAGE.update(input_tokens=count_tokens(system + prompt), output_tokens=count_tokens(out))
    return out

def usage():
    return dict(LAST_USAGE)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, usage, count_tokens, ONLINE, MODEL

## 2. NEW (v4): The SmartDesk test set
Twenty synthetic tickets with the correct category. Both models get exactly the same tickets and the same label set.

In [ ]:
TICKETS = [   # 20 labelled SmartDesk tickets (synthetic)
    ("I was charged twice for my November bill.", "billing"),
    ("Please refund the roaming charge from my trip.", "billing"),
    ("Why is there a late payment fee on my invoice?", "billing"),
    ("My payment failed but the money left my bank.", "billing"),
    ("Can I get a copy of last month's invoice?", "billing"),
    ("No signal at home since Monday.", "network"),
    ("Mobile data is very slow in the evenings.", "network"),
    ("Is there an outage in Osaka today?", "network"),
    ("5G keeps dropping to 4G on my phone.", "network"),
    ("Calls cut out after a few seconds in my office.", "network"),
    ("I cannot log in after resetting my password.", "account"),
    ("Please change the email address on my account.", "account"),
    ("My new SIM card is not activated yet.", "account"),
    ("I want to upgrade my plan to unlimited.", "account"),
    ("Someone else is using my account, please lock it.", "account"),
    ("Do you have a shop near Shibuya station?", "other"),
    ("What are your customer service opening hours?", "other"),
    ("I want to give feedback about your friendly staff.", "other"),
    ("Can I recycle my old phone with you?", "other"),
    ("My bill is fine but the network is terrible lately.", "network"),
]
LABELS = ["billing", "network", "account", "other"]
print(len(TICKETS), "tickets")

## 3. NEW (v4): Quantisation in numbers
Weight memory = parameters x bits / 8. This is weights only; running the model needs extra memory.

In [ ]:
import pandas as pd

def weights_gb(params_billion, bits):
    return params_billion * 1e9 * bits / 8 / 1e9

rows = [{"params": f"{p}B", "fp16": weights_gb(p, 16), "int8": weights_gb(p, 8), "int4": weights_gb(p, 4)}
        for p in [0.5, 1.5, 3, 7]]
print(pd.DataFrame(rows).to_string(index=False))

### LoRA (concept)
LoRA freezes the model and trains two small matrices per layer. How small?

In [ ]:
d, r = 4096, 8
full, lora = d * d, 2 * d * r
print(f"full matrix: {full:,} weights | LoRA rank {r}: {lora:,} weights | {lora / full:.2%}")

## 4. NEW (v4): Load a small open-weight model
The model IDs below are examples from the Qwen family; Phi, Gemma, Llama and Mistral small models work the same way. Check the model card and licence first.

The next cell downloads a model, so the offline check skips it. If it fails or you are offline, the notebook uses the stand-in in section 5.

In [ ]:
import os
slm_pipe, SLM_ID = None, "offline stand-in"
try:
    import torch
    from transformers import pipeline
    HAVE_TRANSFORMERS = os.getenv("OFFLINE") != "1"
except Exception:
    HAVE_TRANSFORMERS = False
print("transformers available:", HAVE_TRANSFORMERS)

In [ ]:
if HAVE_TRANSFORMERS:
    gpu = torch.cuda.is_available()
    SLM_ID = "Qwen/Qwen2.5-1.5B-Instruct" if gpu else "Qwen/Qwen2.5-0.5B-Instruct"   # CPU fallback: smaller model
    try:
        slm_pipe = pipeline("text-generation", model=SLM_ID, device=0 if gpu else -1)
        print("Loaded", SLM_ID, "on", "GPU" if gpu else "CPU")
    except Exception as e:
        slm_pipe, SLM_ID = None, "offline stand-in"
        print("Could not load the model, using the offline stand-in:", type(e).__name__)

## 5. NEW (v4): Two classifiers, one interface
Both return `(label, confidence)`. The offline stand-in is a keyword rule. It is **not** a language model; it only lets the code run without downloads.

In [ ]:
import re, time

KEYWORDS = {"billing": ["bill", "charge", "refund", "invoice", "payment", "fee"],
            "network": ["signal", "network", "slow", "outage", "5g", "4g", "calls", "data"],
            "account": ["password", "log in", "login", "account", "sim", "plan", "email"]}

def offline_slm(text):
    """Offline stand-in for a small model: keyword counts, with a rough confidence."""
    low = text.lower()
    scores = {lab: sum(w in low for w in words) for lab, words in KEYWORDS.items()}
    best = max(scores, key=scores.get)
    total = sum(scores.values())
    if total == 0:
        return "other", 0.6
    return best, round(scores[best] / total, 2)

PROMPT = ("Classify the support ticket into exactly one of: billing, network, account, other. "
          "Reply with one word.\nTicket: {t}")

def classify_slm(text):
    if slm_pipe is None:
        return offline_slm(text)
    out = slm_pipe(PROMPT.format(t=text) + "\nLabel:", max_new_tokens=4, do_sample=False,
                   return_full_text=False)[0]["generated_text"].lower()
    label = next((l for l in LABELS if l in out), None)
    return (label, 0.9) if label else ("other", 0.3)        # unparseable answer = low confidence

def classify_llm(text):
    out = llm(PROMPT.format(t=text), temperature=0).strip().lower()
    label = next((l for l in LABELS if l in out), "other")
    return label, 0.95

print(classify_slm("My payment failed twice"), classify_llm("My payment failed twice"))

## 6. NEW (v4): Compare accuracy, latency and estimated cost
Prices are **placeholders**, not real prices. The SLM cost is a placeholder estimate of your own compute per 1M tokens.

In [ ]:
PRICE_PER_1M = {   # PLACEHOLDER prices, replace with the approved price sheet
    "llm": {"input": 1.00, "output": 4.00},
    "slm": {"input": 0.05, "output": 0.20},   # placeholder estimate of self-hosted compute
}

def est_cost(kind, text, label):
    p = PRICE_PER_1M[kind]
    return (count_tokens(PROMPT.format(t=text)) * p["input"] + count_tokens(label) * p["output"]) / 1_000_000

results = []
for kind, fn in [("slm", classify_slm), ("llm", classify_llm)]:
    for text, gold in TICKETS:
        t0 = time.perf_counter()
        label, conf = fn(text)
        ms = (time.perf_counter() - t0) * 1000
        results.append({"model": kind, "ticket": text, "gold": gold, "pred": label, "conf": conf,
                        "ms": ms, "cost": est_cost(kind, text, label)})
df = pd.DataFrame(results)
summary = df.assign(correct=df.gold == df.pred).groupby("model").agg(
    accuracy=("correct", "mean"), avg_ms=("ms", "mean"), cost_per_1k=("cost", lambda c: c.mean() * 1000))
print("Mode:", "ONLINE" if ONLINE else "OFFLINE demo", "| SLM:", SLM_ID)
print(summary.round(4).to_string())

In [ ]:
wrong = df[df.gold != df.pred][["model", "ticket", "gold", "pred"]]
print(wrong.to_string(index=False) if len(wrong) else "No mistakes")

**TODO:** add five tricky tickets (two topics in one, sarcasm, Japanese text) and run the comparison again.

## 7. NEW (v4): Model routing, SLM first
Use the SLM's confidence. Below the threshold, escalate to the LLM.

In [ ]:
def route(text, threshold=0.7):
    label, conf = classify_slm(text)
    if conf >= threshold:
        return label, "slm", est_cost("slm", text, label)
    label2, _ = classify_llm(text)
    return label2, "llm", est_cost("slm", text, label) + est_cost("llm", text, label2)

rows = []
for th in [0.5, 0.7, 0.9]:
    out = [(route(t, th), gold) for t, gold in TICKETS]
    rows.append({"threshold": th,
                 "escalated": sum(r[1] == "llm" for r, _ in out) / len(out),
                 "accuracy": sum(r[0] == g for r, g in out) / len(out),
                 "cost_per_1k": sum(r[2] for r, _ in out) / len(out) * 1000})
print(pd.DataFrame(rows).round(4).to_string(index=False))

## 8. NEW (v4): Optimisation levers on one prompt
Trim the context, cap the output and keep stable text first. Watch the token count drop.

In [ ]:
RULES = "You are SmartDesk, a polite telecom support assistant. Follow the refund policy. " * 20
FAQ = ["Refunds are possible within 14 days.", "Roaming charges appear on the next bill.",
       "Outages are posted on the status page.", "Invoices are in the app under Billing.",
       "Password resets expire after 30 minutes.", "SIM activation takes up to 2 hours."]
ticket = "Please refund the roaming charge from my trip to Seoul."

def build_prompt(top_k, max_words_rules):
    rules = " ".join(RULES.split()[:max_words_rules])
    return f"{rules}\n\nFAQ:\n" + "\n".join(FAQ[:top_k]) + f"\n\nTicket: {ticket}"

for name, k, words in [("baseline", 6, 400), ("top-k 3", 3, 400), ("top-k 3 + trimmed rules", 3, 60)]:
    print(f"{name:<24} {count_tokens(build_prompt(k, words)):>5} input tokens")
print("Output limit: pass max_tokens=150 to llm() to cap the reply length.")

Next: open `Day25_Token_Cost_Calculator_Colab.ipynb` to turn these savings into cost per 1,000 tickets.

## 9. Check yourself
1. Which model made more mistakes, and on which kind of ticket?
2. What threshold would you choose for SmartDesk, and why?
3. Which numbers in this notebook are placeholders?

---
**Ganbatte (頑張って): *Do your best***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)